# NLP Communication AnalysisConverted from `src/nlp_analysis.py`---

**Beschreibung:** NLP Analysis - Sentiment and communication quality

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# VADER Sentiment (optional import)
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_AVAILABLE = True
except ImportError:
    VADER_AVAILABLE = False
    print("VADER nicht installiert. Installiere mit: pip install vaderSentiment")

# Word lists for pattern recognition
POLITENESS_WORDS = [
    'please', 'thank', 'thanks', 'appreciate', 'grateful',
    'sorry', 'apolog', 'kindly', 'would you', 'could you'
]

URGENCY_WORDS = [
    'urgent', 'asap', 'immediately', 'critical', 'emergency',
    'deadline', 'priority', 'important', 'blocker', 'blocked'
]

TECHNICAL_WORDS = [
    'error', 'bug', 'fix', 'issue', 'problem', 'crash',
    'exception', 'failed', 'timeout', 'null', 'undefined'
]

SOLUTION_WORDS = [
    'fixed', 'resolved', 'solved', 'solution', 'working',
    'deployed', 'released', 'updated', 'patched', 'done'
]


def analyze_sentiment(text):
    """
    Analyze das Sentiment eines Textes mit VADER (falls verfügbar).
    Returns:
        dict: Sentiment-Scores (compound, pos, neg, neu)
    """
    if not VADER_AVAILABLE:
        return {'compound': 0.0, 'pos': 0.0, 'neg': 0.0, 'neu': 1.0}
    
    if pd.isna(text) or not str(text).strip():
        return {'compound': 0.0, 'pos': 0.0, 'neg': 0.0, 'neu': 1.0}
    
    analyzer = SentimentIntensityAnalyzer()
    return analyzer.polarity_scores(str(text))


def extract_text_features(text):
    """Extrahiert Basis-Features aus Text."""
    if pd.isna(text) or not str(text).strip():
        return {
            'word_count': 0,
            'char_count': 0,
            'question_count': 0,
            'exclamation_count': 0
        }
    
    text = str(text)
    words = text.split()
    return {
        'word_count': len(words),
        'char_count': len(text),
        'question_count': text.count('?'),
        'exclamation_count': text.count('!')
    }


def extract_patterns(text):
    """Count communication patterns in text (keyword matching)."""
    if pd.isna(text) or not str(text).strip():
        return {
            'politeness_score': 0,
            'urgency_score': 0,
            'technical_score': 0,
            'solution_score': 0
        }
    
    text_lower = str(text).lower()
    return {
        'politeness_score': sum(1 for w in POLITENESS_WORDS if w in text_lower),
        'urgency_score':    sum(1 for w in URGENCY_WORDS    if w in text_lower),
        'technical_score':  sum(1 for w in TECHNICAL_WORDS  if w in text_lower),
        'solution_score':   sum(1 for w in SOLUTION_WORDS   if w in text_lower)
    }


def process_utterances(utterances_df):
    """
    Verarbeitet alle Utterances/Kommentare und extrahiert NLP-Features.
    
    Args:
        utterances_df: DataFrame mit Kommentaren (mind. Spalten: issueid, actionbody/body)
    
    Returns:
        DataFrame mit NLP-Features pro Kommentar
    """
    print("Verarbeite Kommentare...")
    
    # Flexible Text-Spaltenname
    text_col = 'actionbody' if 'actionbody' in utterances_df.columns else 'body'
    if text_col not in utterances_df.columns:
        raise ValueError(f"Keine Textspalte gefunden (erwartet: 'actionbody' oder 'body')")
    
    results = []
    total = len(utterances_df)
    
    for idx, row in utterances_df.iterrows():
        text = row[text_col] if pd.notna(row[text_col]) else ""
        
        # Sentiment
        sentiment = analyze_sentiment(text)
        
        # Text-Features
        text_feat = extract_text_features(text)
        
        # Keyword-Patterns
        patterns = extract_patterns(text)
        
        results.append({
            'issueid': row.get('issueid', idx),
            'author_role': row.get('author_role', 'unknown'),
            'sentiment_compound': sentiment['compound'],
            'sentiment_pos': sentiment['pos'],
            'sentiment_neg': sentiment['neg'],
            **text_feat,
            **patterns
        })
        
        # Fortschritt (nur alle 5000 Zeilen)
        if (idx + 1) % 5000 == 0:
            print(f"   {idx+1:,} / {total:,} verarbeitet...")
    
    print(f"{len(results):,} Kommentare analysiert")
    return pd.DataFrame(results)


def aggregate_by_issue(features_df):
    """
    Aggregiert NLP-Features pro Issue (Mittelwerte, Summen, etc.).
    
    Returns:
        DataFrame mit aggregierten Metriken pro issueid
    """
    print("Aggregiere pro Issue...")
    
    agg_dict = {
        'sentiment_compound': ['mean', 'std', 'min', 'max'],
        'sentiment_pos':      'mean',
        'sentiment_neg':      'mean',
        'word_count':         ['mean', 'sum'],
        'question_count':     'sum',
        'politeness_score':   'sum',
        'urgency_score':      'sum',
        'technical_score':    'sum',
        'solution_score':     'sum'
    }
    
    aggregated = features_df.groupby('issueid').agg(agg_dict).reset_index()
    
    # MultiIndex → flache Spaltennamen
    aggregated.columns = ['_'.join(col).strip('_') for col in aggregated.columns.values]
    
    print(f"{len(aggregated):,} Issues aggregiert")
    return aggregated

VADER nicht installiert. Installiere mit: pip install vaderSentiment


##  Execution

In [4]:
import pandas as pd
from pathlib import Path

print("=" * 50)
print(" NLP-ANALYSE")
print("=" * 50)

# ─── Load utterances / comments ───
data_path = Path("data/raw/sample_utterances.csv")

if data_path.exists():
    utterances = pd.read_csv(data_path)
    print(f" Loaded: {len(utterances):,} Kommentare")
    
    # 1. Extract NLP features per single comment
    features = process_utterances(utterances)
    
    # 2. Aggregate per issue/ticket
    issue_features = aggregate_by_issue(features)
    
    # 3. Save processed result
    output_path = Path("data/processed/nlp_features.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    issue_features.to_csv(output_path, index=False)
    print(f"\n Saved: {output_path}")
    
    # 4. Quick summary statistics
    if 'sentiment_compound_mean' in issue_features.columns:
        avg_sentiment = issue_features['sentiment_compound_mean'].mean()
        print(f"\n Durchschnittliches Sentiment pro Issue: {avg_sentiment:.3f}")
        
        # Optional: more interesting stats
        print(f" Anzahl Issues mit sehr negativem Sentiment (< -0.3): "
              f"{(issue_features['sentiment_compound_mean'] < -0.3).sum()}")
        print(f" Anzahl Issues mit hoher Höflichkeit (≥ 3): "
              f"{(issue_features['politeness_score_sum'] >= 3).sum()}")
else:
    print(" Utterances-Datei nicht gefunden!")
    print(f" Gesuchter Pfad: {data_path.absolute()}")
    print(" → Prüfe Ordnerstruktur, Dateiname und aktuellen Arbeitsverzeichnis.")

 NLP-ANALYSE
 Utterances-Datei nicht gefunden!
 Gesuchter Pfad: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/sample_utterances.csv
 → Prüfe Ordnerstruktur, Dateiname und aktuellen Arbeitsverzeichnis.
